# T10 It5: pre-registration of the finalists (before NUDT 808 is read)

NUDT fresh 808 (300 photos, pool 2400, the 500-pool and every 404 / 505 / 606 / 707 photo excluded) is being dumped by `run_gpu.sh`; nothing
here reads it. Finalists, all trained on the NUDT design sets only (101, 202, 404, 505, 606), cut by Learn-then-Test (alpha 0.005, floor 0.9)
on those sets unless fixed:

| id | machine | rule | why it is a finalist |
|---|---|---|---|
| base | PC / Jetson | bar 8 / 11 (PC); bar 8, cap 10, M = 5 (Jetson today) | reference |
| J0 | both | T8 judge, override 12 (Jetson: early exit 0.98, context from 5) | the T8 recommendation |
| J1 | both | J0 without the override | T9: override caused most wrong answers under inflation, cost 0 on clean data |
| J2 | both | J1 with the cut fixed at 0.98 | T9: the only rule with 0 wrong in every altered run (Jetson) |
| V5 | both | relative inliers + scale augmentation, no override | T10 It2: wrong answers flat across inlier scales |

Scored once in It6 on NUDT 808 at x1 (primary) and with every inlier signal scaled x0.5, 0.75, 1.25, 1.5, 2 (robustness).

In [ ]:
import os, sys, json, hashlib, datetime
os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd, gc
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
from train.checks.holdout import NUDT_DESIGN, load_t8_table
RES = ROOT / "tests/Test_Iteration_10/results"
D = load_t8_table(ROOT, locked=True); D = D[D.data.isin(NUDT_DESIGN)].reset_index(drop=True)
assert not (RES / "signals" / "nudt808.csv.gz").exists() or True     # the 808 table may exist; it is not read here
AUG = [0.5, 0.75, 1.0, 1.5, 2.0]; F = cj.FEATURES_CTX; FR = [("rli" if f == "li" else "rln22" if f == "ln22" else f) for f in F]
def ltt(table, p, sets, **kw):
    t = cj.ltt_tau(cj.scan_tau(table, p, cj.TAUS, **kw), sets, 0.005)
    return None if t is None else max(t, 0.9)
CUTS = {}
for var, m in [("pc", None), ("jetson", 5)]:
    t = D if m is None else cj.add_context(D, m=m)
    p, _ = cj.judge_probability(t, NUDT_DESIGN, F, "gbm")
    kw = dict(min_choice=3, wide_tau=0.95, **(dict(jetson=True, m=5) if m else {}))
    CUTS[(var, "J0")] = ltt(t, p, NUDT_DESIGN, override=12, **kw)
    CUTS[(var, "J1")] = ltt(t, p, NUDT_DESIGN, override=None, **kw)
    lv = sr.far_levels(t, m); prior = dict(far_inl=float(lv.far_inl.mean()), far_n22=float(lv.far_n22.mean()))
    model, p_oof, st = sr.fit_judge(t, NUDT_DESIGN, FR, {"rli", "z", "hull", "cover9"}, AUG, lambda x: sr.add_relative(x, prior, 30, m))
    names = st.data + "_x" + st.scale_factor.astype(str)
    CUTS[(var, "V5")] = ltt(st.assign(data=names), p_oof, list(names.unique()), override=None, **kw)
    CUTS[(var, "V5_prior")] = prior
    del st, p_oof; gc.collect()
CUTS

In [ ]:
PRE = dict(written=datetime.datetime.now().isoformat(timespec="seconds"), final_set="nudt808",
    note="NUDT fresh 808: 300 photos, pool 2400, the 500-pool and every 404/505/606/707 photo excluded; dumped by run_gpu.sh before any T10 rule "
         "was chosen; read only by It6. Official TEST and NUDT 707 are seen data (T8) and are reported only as context.",
    judges_trained_on=NUDT_DESIGN, features="FEATURES_CTX (V5: li -> rli, ln22 -> rln22; prior weight 30)", jetson_context="rank < 5",
    finalists={var: {"base": "bar 8 / 11" if var == "pc" else "bar 8, cap 10, M 5",
                     "J0": dict(cut=CUTS[(var, "J0")], override=12, wide_cut=0.95, min_choice=3, early_exit=0.98 if var == "jetson" else None),
                     "J1": dict(cut=CUTS[(var, "J1")], override=None, wide_cut=0.95, min_choice=3, early_exit=0.98 if var == "jetson" else None),
                     "J2": dict(cut=0.98, override=None, wide_cut=0.98, min_choice=3, early_exit=0.98 if var == "jetson" else None),
                     "V5": dict(cut=CUTS[(var, "V5")], override=None, wide_cut=0.95, min_choice=3, early_exit=0.98 if var == "jetson" else None,
                                augment=AUG, prior=CUTS[(var, "V5_prior")])} for var in ["pc", "jetson"]},
    primary="NUDT 808 at x1: a finalist counts as a gain over its base if it has no more wrong answers than the base, 0 wrong, gain >= 0 and a "
            "paired-bootstrap 95 % interval above 0. A finalist counts as SAFER than J0 if it has fewer wrong answers on 808 over the scale range.",
    robustness="NUDT 808 with every inlier-count signal scaled x0.5, 0.75, 1.25, 1.5, 2: wrong answers per scale; a variant is scale-robust only "
               "with 0 wrong at every scale.")
path = RES / "preregistered_t10.json"
assert not path.exists(), "pre-registration already written"
path.write_text(json.dumps(PRE, indent=1, default=str)); print(path.read_text()); print("sha256", hashlib.sha256(path.read_bytes()).hexdigest())